In [2]:
import sys,os,glob
import json
from openbabel import openbabel
# import cifutils
openbabel.obErrorLog.SetOutputLevel(0)


In [17]:
openbabel.

AttributeError: module 'openbabel.openbabel' has no attribute 'OBElementTable'

In [3]:
sdfnames = glob.glob('/projects/ml/ligand_datasets/pdb/ligands/?/*_model.sdf')
len(sdfnames)

39329

In [12]:
%%time
obConversion = openbabel.OBConversion()
obConversion.SetInFormat("sdf")
obConversion.SetOutFormat("sdf")
ligands = {}
for sdfname in sdfnames[:]:
    obmol = openbabel.OBMol()
    obConversion.ReadFile(obmol,sdfname)
    xyz = np.array([(a.x(),a.y(),a.z()) for a in openbabel.OBMolAtomIter(obmol)])

    cifname = sdfname.replace('_model.sdf','.cif')
    try:
        cif = cifutils.ParsePDBLigand(cifname)
    except:
        print("FAILED:", sdfname)
        continue

    if obmol.NumAtoms()!=cif['xyz'].shape[0]:
        print("FAILED:", sdfname)
        continue
    
    flag = ((xyz-cif['xyz'])[~np.isnan(cif['xyz'])]<1e-3).all()
    if flag==False:
        print("FAILED:", sdfname)
        continue

    ID = cifname.split('/')[-1][:-4]
    ligands[ID] = {
        'sdf' : obConversion.WriteString(obmol),
        'atom_id' : cif['atom_id'].tolist(),
        'leaving' : cif['leaving'].tolist(),
        'pdbx_align' : cif['pdbx_align'].tolist()
    }

*** Open Babel Error  in TetStereoToWedgeHash
  Failed to set stereochemistry as unable to find an available bond
*** Open Babel Error  in TetStereoToWedgeHash
  Failed to set stereochemistry as unable to find an available bond
*** Open Babel Error  in TetStereoToWedgeHash
  Failed to set stereochemistry as unable to find an available bond
*** Open Babel Error  in TetStereoToWedgeHash
  Failed to set stereochemistry as unable to find an available bond
*** Open Babel Error  in TetStereoToWedgeHash
  Failed to set stereochemistry as unable to find an available bond
*** Open Babel Error  in TetStereoToWedgeHash
  Failed to set stereochemistry as unable to find an available bond
*** Open Babel Error  in TetStereoToWedgeHash
  Failed to set stereochemistry as unable to find an available bond
*** Open Babel Error  in TetStereoToWedgeHash
  Failed to set stereochemistry as unable to find an available bond
*** Open Babel Error  in TetStereoToWedgeHash
  Failed to set stereochemistry as unable 

FAILED: /projects/ml/ligand_datasets/pdb/ligands/U/UNL_model.sdf
FAILED: /projects/ml/ligand_datasets/pdb/ligands/U/UJW_model.sdf


*** Open Babel Error  in TetStereoToWedgeHash
  Failed to set stereochemistry as unable to find an available bond
*** Open Babel Error  in TetStereoToWedgeHash
  Failed to set stereochemistry as unable to find an available bond


CPU times: user 2min 6s, sys: 4.68 s, total: 2min 11s
Wall time: 12min 41s


In [13]:
out_str = '{\n'
for k,v in ligands.items():
    out_str += '\t"%s" : %s,\n'%(k,json.dumps(v))
out_str = out_str[:-2] + '\n}\n'

with open("ligands.json", "w") as outfile:
    outfile.write(out_str)


In [15]:
!rm -f ligands.json.gz
!gzip ligands.json

In [16]:
%%time
with gzip.open('ligands.json.gz','rt') as file:
    ligands = json.load(file)
len(ligands)

CPU times: user 2.24 s, sys: 463 ms, total: 2.7 s
Wall time: 2.76 s


39096